# Gold — dim_demografia

UNION DISTINCT de combinaciones `(sexo, grupo_etario_label)` de todas las fuentes Silver.
Mapea cada etiqueta a rangos numéricos `(min, max)` usando el catálogo `ref_grupos_etarios`.
Para grupos etarios numéricos de WHO (ej. `25-29`, `85+`) aplica parsing por expresión regular.

| Columna | Tipo | Descripción |
|---|---|---|
| `demografia_sk` | INT | Surrogate key |
| `sexo` | STRING | `M` \| `F` \| `Ambos` |
| `grupo_etario_label` | STRING | Etiqueta original del grupo |
| `grupo_etario_anios_min` | DOUBLE | Edad mínima en años (NULL si no aplica) |
| `grupo_etario_anios_max` | DOUBLE | Edad máxima en años (NULL si sin límite superior) |

**Fila placeholder obligatoria:** `(sexo='Ambos', grupo_etario_label='Sin desagregar', min=NULL, max=NULL)` — para INE-geo que no tiene desagregación etaria.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, StringType
from functools import reduce
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

SILVER_SCHEMA = 'stage_silver_ss2'
GOLD_SCHEMA   = 'gold_ss2'
WRITE_MODE    = 'overwrite'

def get_job_run_id():
    try:
        return (
            dbutils.notebook.entry_point
            .getDbutils().notebook().getContext()
            .currentRunId().toString()
        )
    except Exception:
        return 'manual'

RUN_ID = get_job_run_id()
logger.info(f'Setup OK — run_id={RUN_ID}')

## Catálogo de referencia — grupos etarios

Datos de `ref/ref_grupos_etarios.csv` inlineados. Cubre MSPAS, INE, grupos materno-infantil,
neonatales/postneonatales y etiquetas especiales.

In [ ]:
_REF_GRUPOS = [
    ('<1 año',           0.0,    1.0),
    ('1 a 4 años',       1.0,    4.0),
    ('5 a 9 años',       5.0,    9.0),
    ('10 a 14 años',    10.0,   14.0),
    ('15 a 19 años',    15.0,   19.0),
    ('20 a 24 años',    20.0,   24.0),
    ('25 a 29 años',    25.0,   29.0),
    ('30 a 34 años',    30.0,   34.0),
    ('35 a 39 años',    35.0,   39.0),
    ('40 a 44 años',    40.0,   44.0),
    ('45 a 49 años',    45.0,   49.0),
    ('50 a 54 años',    50.0,   54.0),
    ('55 a 59 años',    55.0,   59.0),
    ('60 a 64 años',    60.0,   64.0),
    ('65 a 69 años',    65.0,   69.0),
    ('70 a 74 años',    70.0,   74.0),
    ('75 a 79 años',    75.0,   79.0),
    ('80 a 84 años',    80.0,   84.0),
    ('85 años y más',   85.0,   None),
    ('0 - 4 años',       0.0,    4.0),
    ('5 - 9 años',       5.0,    9.0),
    ('10 - 14 años',    10.0,   14.0),
    ('15 - 19 años',    15.0,   19.0),
    ('20 - 24 años',    20.0,   24.0),
    ('25 - 29 años',    25.0,   29.0),
    ('30 - 34 años',    30.0,   34.0),
    ('35 - 39 años',    35.0,   39.0),
    ('40 - 44 años',    40.0,   44.0),
    ('45 - 49 años',    45.0,   49.0),
    ('50 - 54 años',    50.0,   54.0),
    ('55 - 59 años',    55.0,   59.0),
    ('60 - 64 años',    60.0,   64.0),
    ('65 - 69 años',    65.0,   69.0),
    ('70 - 74 años',    70.0,   74.0),
    ('75 - 79 años',    75.0,   79.0),
    ('80 - 84 años',    80.0,   84.0),
    ('75 años y más',   75.0,   None),
    ('<1 mes',           0.0,    0.08),
    ('1m a <2m',         0.08,   0.17),
    ('2m a <1a',         0.17,   1.0),
    ('0 días',           0.0,    0.003),
    ('1 día',            0.003,  0.005),
    ('2 días',           0.005,  0.008),
    ('3 días',           0.008,  0.011),
    ('4 días',           0.011,  0.014),
    ('5 días',           0.014,  0.016),
    ('6 días',           0.016,  0.019),
    ('7 días',           0.019,  0.022),
    ('8 días',           0.022,  0.025),
    ('9 días',           0.025,  0.027),
    ('10 días',          0.027,  0.030),
    ('11 días',          0.030,  0.033),
    ('12 días',          0.033,  0.036),
    ('13 días',          0.036,  0.038),
    ('14 días',          0.038,  0.041),
    ('15 días',          0.041,  0.044),
    ('16 días',          0.044,  0.047),
    ('17 días',          0.047,  0.049),
    ('18 días',          0.049,  0.052),
    ('19 días',          0.052,  0.055),
    ('20 días',          0.055,  0.058),
    ('21 días',          0.058,  0.060),
    ('22 días',          0.060,  0.063),
    ('23 días',          0.063,  0.066),
    ('24 días',          0.066,  0.068),
    ('25 días',          0.068,  0.071),
    ('26 días',          0.071,  0.074),
    ('27 días',          0.074,  0.077),
    ('28 días',          0.077,  0.082),
    ('1 mes',            0.08,   0.17),
    ('2 meses',          0.17,   0.25),
    ('3 meses',          0.25,   0.33),
    ('4 meses',          0.33,   0.42),
    ('5 meses',          0.42,   0.50),
    ('6 meses',          0.50,   0.58),
    ('7 meses',          0.58,   0.67),
    ('8 meses',          0.67,   0.75),
    ('9 meses',          0.75,   0.83),
    ('10 meses',         0.83,   0.92),
    ('11 meses',         0.92,   1.0),
    ('Todas las edades', None,   None),
    ('Todas',            None,   None),
    ('Todos los grupos', None,   None),
    ('Otras edades',     None,   None),
    ('Otros',            None,   None),
    ('No especificado',  None,   None),
    ('Sin desagregar',   None,   None),
]

ref_grupos = spark.createDataFrame(
    _REF_GRUPOS,
    ['grupo_etario_label', 'grupo_etario_anios_min', 'grupo_etario_anios_max']
)
logger.info(f'Catálogo ref_grupos: {ref_grupos.count()} entradas')

## Recolección de etiquetas por fuente

In [ ]:
# INE-edad: sexo='Ambos' (hombres/mujeres son medidas, no dimensión), grupo desde col 'edad'
_INE_EDAD = ['ine_defunciones_sexo_edad', 'ine_defunciones_neonatales', 'ine_defunciones_postneonatales']
df_ine_edad = (
    reduce(lambda a, b: a.union(b), [
        spark.table(f'{SILVER_SCHEMA}.{t}').select(F.col('edad').alias('grupo_etario_label'))
        for t in _INE_EDAD
    ])
    .distinct()
    .withColumn('sexo', F.lit('Ambos'))
)

# INE-geo: sexo='Ambos', grupo='Sin desagregar' (sin desagregación etaria)
df_ine_geo = spark.createDataFrame(
    [('Ambos', 'Sin desagregar')],
    ['sexo', 'grupo_etario_label']
)

# MSPAS: sexo (M/F), grupo_etario. Filtra sexo=NULL — esas filas van al placeholder en fact_morbilidad.
_MSPAS = [
    'dbx_primeras_causas_de_morbilidad_2015_a_2025',
    'dbx_morbilidad_enfermedades_cronicas_2015_a_2025',
    'dbx_morbilidad_grupo_materno_infantil_2012_a_2025',
]
df_mspas = (
    reduce(lambda a, b: a.union(b), [
        spark.table(f'{SILVER_SCHEMA}.{t}').select('sexo', F.col('grupo_etario').alias('grupo_etario_label'))
        for t in _MSPAS
    ])
    .filter(F.col('sexo').isNotNull())
    .distinct()
)

# WHO: sexo (M/F/Ambos), grupo_etario normalizado ('Todas', '25-29', '85+', ...)
df_who = (
    reduce(lambda a, b: a.union(b), [
        spark.table(f'{SILVER_SCHEMA}.{t}').select('sexo', F.col('grupo_etario').alias('grupo_etario_label'))
        for t in ['who_guatemala', 'who_costa_rica']
    ])
    .distinct()
)

df_all_labels = (
    df_ine_edad.select('sexo', 'grupo_etario_label')
    .union(df_ine_geo)
    .union(df_mspas)
    .union(df_who)
    .dropDuplicates()
)

logger.info(f'Combinaciones (sexo, grupo_etario_label) únicas: {df_all_labels.count()}')

## Join con ref y parsing de rangos numéricos WHO

Para etiquetas de WHO con formato numérico (`25-29`, `85+`) que no están en el catálogo,
se parsean con expresiones regulares (sin backslash — usa clases de caracteres `[0-9]`).

In [ ]:
df_joined = df_all_labels.join(ref_grupos, 'grupo_etario_label', 'left')

# Parsear formato '25-29' → min=25.0, max=29.0
df_parsed = (
    df_joined
    .withColumn(
        'grupo_etario_anios_min',
        F.when(
            F.col('grupo_etario_anios_min').isNull() & F.col('grupo_etario_label').rlike('^[0-9]+-[0-9]+$'),
            F.split(F.col('grupo_etario_label'), '-').getItem(0).cast(DoubleType())
        ).when(
            F.col('grupo_etario_anios_min').isNull() & F.col('grupo_etario_label').rlike('^[0-9]+[+]$'),
            F.regexp_extract(F.col('grupo_etario_label'), '^([0-9]+)[+]$', 1).cast(DoubleType())
        ).otherwise(F.col('grupo_etario_anios_min'))
    )
    .withColumn(
        'grupo_etario_anios_max',
        F.when(
            F.col('grupo_etario_anios_max').isNull() & F.col('grupo_etario_label').rlike('^[0-9]+-[0-9]+$'),
            F.split(F.col('grupo_etario_label'), '-').getItem(1).cast(DoubleType())
        ).otherwise(F.col('grupo_etario_anios_max'))
    )
)

df_dim = df_parsed.withColumn(
    'demografia_sk',
    F.row_number().over(Window.orderBy('sexo', 'grupo_etario_label'))
)

df_dim = df_dim.select(
    'demografia_sk', 'sexo', 'grupo_etario_label',
    'grupo_etario_anios_min', 'grupo_etario_anios_max'
)

logger.info(f'dim_demografia: {df_dim.count()} combinaciones únicas')

df_dim.write \
    .format('delta') \
    .mode(WRITE_MODE) \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{GOLD_SCHEMA}.dim_demografia')

logger.info(f'Escrito → {GOLD_SCHEMA}.dim_demografia')

## Validación

In [ ]:
df_val = spark.table(f'{GOLD_SCHEMA}.dim_demografia')
print(f'Total combinaciones: {df_val.count()}')

print('\n── Distribución por sexo ──')
df_val.groupBy('sexo').count().show()

print('\n── Verificar placeholder Sin desagregar ──')
placeholder = df_val.filter(
    (F.col('sexo') == 'Ambos') & (F.col('grupo_etario_label') == 'Sin desagregar')
).count()
print(f'Placeholder (Ambos / Sin desagregar): {placeholder} fila (debe ser 1)')

print('\n── Etiquetas sin mapeo min/max (aparte de especiales) ──')
sin_mapeo = df_val.filter(
    F.col('grupo_etario_anios_min').isNull() &
    ~F.col('grupo_etario_label').isin(['Sin desagregar', 'Todas', 'Todas las edades', 'Todos los grupos', 'Otras edades', 'Otros', 'No especificado'])
)
sin_mapeo.show(truncate=False)

print('\n── Muestra ──')
df_val.orderBy('sexo', 'grupo_etario_anios_min').show(20, truncate=False)